# CardioGraph - Echocardiogram ejection fraction (EchoNet-Dynamic, R(2+1)D-18)

Training pipeline for the CardioGraph **EF extractor**: predicts left-ventricular ejection fraction (EF %) from an
apical-4-chamber echo cine loop and maps it to a clinical category.

| | |
|---|---|
| **Task** | Video regression: EF % (0-100) |
| **Data** | EchoNet-Dynamic, 10,030 A4C echo videos (112 x 112, AVI) with expert EF labels, official TRAIN/VAL/TEST split |
| **Input** | 32-frame clip, every 2nd frame (~1.3 s at 50 fps), normalised with training-set channel mean/std |
| **Model** | `torchvision` R(2+1)D-18, Kinetics-400 pretrained, single regression output |
| **Eval** | Several evenly spaced clips per video, averaged; MAE / RMSE / R2 and AUC for EF < 50 % |
| **Output** | `r2plus1d_18_kinetics_echonet_ef_32f_p2_best.pth` |

Clip length, sampling period, augmentation and backbone follow the EchoNet-Dynamic paper (Ouyang et al., *Nature* 2020).

> Rebuilt pipeline (written by Claude) for the CardioGraph Echo model; the original Kaggle notebook was not kept.

**Setup on Kaggle:** upload `EchoNet-Dynamic.zip` (Stanford AIMI, research-use licence) as a private dataset and turn on a GPU.
The folder containing `FileList.csv` and `Videos/` is found automatically.

## Imports & config

In [ ]:
import os, random, time
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision.models.video import r2plus1d_18, R2Plus1D_18_Weights
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, roc_auc_score
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

# SMOKE_TEST=1 runs a tiny CPU pass to check the pipeline end to end (no pretrained download)
SMOKE_TEST = os.environ.get("SMOKE_TEST") == "1"


def find_data_root():
    if os.environ.get("ECHONET_ROOT"):
        return Path(os.environ["ECHONET_ROOT"])
    for base in [Path("/kaggle/input"), Path(".")]:
        if base.exists():
            for hit in sorted(base.rglob("FileList.csv")):
                if (hit.parent / "Videos").is_dir():
                    return hit.parent
    raise FileNotFoundError("FileList.csv + Videos/ not found - attach the EchoNet-Dynamic dataset")


class CFG:
    DATA_ROOT = find_data_root()
    VIDEO_DIR = DATA_ROOT / "Videos"
    OUT_DIR = Path(os.environ.get("OUT_DIR", "/kaggle/working" if Path("/kaggle/working").exists() else "outputs"))

    FRAMES = 32
    PERIOD = 2
    IMAGE_SIZE = 112
    PAD = 12                    # random pad-and-crop augmentation (pixels)
    PRETRAINED = not SMOKE_TEST

    EPOCHS = 1 if SMOKE_TEST else 25
    BATCH_SIZE = 2 if SMOKE_TEST else 16
    LR = 1e-4
    WEIGHT_DECAY = 1e-4
    EARLY_STOP_PATIENCE = 6
    TEST_CLIPS = 2 if SMOKE_TEST else 5
    NORM_VIDEOS = 4 if SMOKE_TEST else 200   # videos used to estimate mean/std

    REDUCED_EF = 50.0           # EF below this = reduced (used for AUC)
    NUM_WORKERS = 0 if SMOKE_TEST else 4
    SEED = 42
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    CKPT_NAME = "r2plus1d_18_kinetics_echonet_ef_32f_p2_best.pth"


CFG.OUT_DIR.mkdir(parents=True, exist_ok=True)
USE_AMP = CFG.DEVICE == "cuda"


def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)


set_seed(CFG.SEED)
print("Data root:", CFG.DATA_ROOT)
print("Device   :", CFG.DEVICE, "| smoke test:", SMOKE_TEST)

## File list & splits

In [ ]:
files = pd.read_csv(CFG.DATA_ROOT / "FileList.csv")
files["FileName"] = files["FileName"].astype(str).apply(lambda f: f if f.lower().endswith(".avi") else f + ".avi")
files["Split"] = files["Split"].str.upper()
files["exists"] = files["FileName"].apply(lambda f: (CFG.VIDEO_DIR / f).exists())
missing = (~files["exists"]).sum()
if missing:
    print(f"Skipping {missing} rows whose video is missing")
files = files[files["exists"]].reset_index(drop=True)

train_df = files[files["Split"] == "TRAIN"].reset_index(drop=True)
val_df = files[files["Split"] == "VAL"].reset_index(drop=True)
test_df = files[files["Split"] == "TEST"].reset_index(drop=True)
print(f"train {len(train_df)} | val {len(val_df)} | test {len(test_df)}")
print(files.groupby("Split")["EF"].describe().round(1))

## Video loading & normalisation

In [ ]:
def load_video(path):
    """AVI -> float32 array (frames, H, W, 3), RGB, resized to IMAGE_SIZE if needed."""
    cap = cv2.VideoCapture(str(path))
    frames = []
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        if frame.shape[:2] != (CFG.IMAGE_SIZE, CFG.IMAGE_SIZE):
            frame = cv2.resize(frame, (CFG.IMAGE_SIZE, CFG.IMAGE_SIZE), interpolation=cv2.INTER_AREA)
        frames.append(frame)
    cap.release()
    if not frames:
        raise ValueError(f"Could not read video: {path}")
    return np.stack(frames).astype(np.float32)


# Channel mean/std from a random sample of training videos (as in EchoNet-Dynamic)
rng = np.random.default_rng(CFG.SEED)
sample = rng.choice(len(train_df), size=min(CFG.NORM_VIDEOS, len(train_df)), replace=False)
s1, s2, n = np.zeros(3), np.zeros(3), 0
for i in tqdm(sample, desc="mean/std"):
    v = load_video(CFG.VIDEO_DIR / train_df.loc[i, "FileName"]).reshape(-1, 3)
    s1 += v.sum(0); s2 += (v ** 2).sum(0); n += len(v)
MEAN = (s1 / n).astype(np.float32)
STD = np.sqrt(s2 / n - MEAN ** 2).astype(np.float32)
print("mean", MEAN.round(2), "std", STD.round(2))

In [ ]:
CLIP_SPAN = (CFG.FRAMES - 1) * CFG.PERIOD + 1


def clip_from(video, start):
    idx = np.clip(start + np.arange(CFG.FRAMES) * CFG.PERIOD, 0, len(video) - 1)   # short videos repeat last frame
    clip = (video[idx] - MEAN) / STD                                                # (F, H, W, 3)
    return torch.from_numpy(np.ascontiguousarray(clip.transpose(3, 0, 1, 2)))      # (3, F, H, W)


def random_pad_crop(clip):
    c, f, h, w = clip.shape
    padded = torch.zeros((c, f, h + 2 * CFG.PAD, w + 2 * CFG.PAD), dtype=clip.dtype)
    padded[:, :, CFG.PAD:CFG.PAD + h, CFG.PAD:CFG.PAD + w] = clip
    y, x = np.random.randint(0, 2 * CFG.PAD + 1, size=2)
    return padded[:, :, y:y + h, x:x + w]


def clip_starts(n_frames, k):
    max_start = max(0, n_frames - CLIP_SPAN)
    return np.linspace(0, max_start, k).round().astype(int)


class EchoDataset(Dataset):
    """train: one random clip + augmentation. eval: k evenly spaced clips -> (k, 3, F, H, W)."""

    def __init__(self, df, train=False, clips=1):
        self.df, self.train, self.clips = df, train, clips

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        video = load_video(CFG.VIDEO_DIR / row["FileName"])
        ef = torch.tensor(row["EF"], dtype=torch.float32)
        if self.train:
            start = np.random.randint(0, max(0, len(video) - CLIP_SPAN) + 1)
            return random_pad_crop(clip_from(video, start)), ef
        clips = torch.stack([clip_from(video, s) for s in clip_starts(len(video), self.clips)])
        return clips, ef


loader_kw = dict(num_workers=CFG.NUM_WORKERS, pin_memory=CFG.DEVICE == "cuda")
train_loader = DataLoader(EchoDataset(train_df, train=True), batch_size=CFG.BATCH_SIZE, shuffle=True,
                          drop_last=len(train_df) > CFG.BATCH_SIZE, **loader_kw)
val_loader = DataLoader(EchoDataset(val_df, clips=1), batch_size=CFG.BATCH_SIZE, **loader_kw)
test_loader = DataLoader(EchoDataset(test_df, clips=CFG.TEST_CLIPS), batch_size=max(1, CFG.BATCH_SIZE // 2), **loader_kw)

xb, yb = next(iter(train_loader))
print("Train batch:", tuple(xb.shape), "EF", yb[:4].tolist())

## Model

In [ ]:
weights = R2Plus1D_18_Weights.KINETICS400_V1 if CFG.PRETRAINED else None
model = r2plus1d_18(weights=weights)
model.fc = nn.Linear(model.fc.in_features, 1)
EF_MEAN = float(train_df["EF"].mean())
nn.init.constant_(model.fc.bias, EF_MEAN)       # start predictions at the mean EF
model = model.to(CFG.DEVICE)

criterion = nn.MSELoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=CFG.LR, weight_decay=CFG.WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG.EPOCHS)
scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
print(f"R(2+1)D-18: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M params | initial bias {EF_MEAN:.1f}")

In [ ]:
def ef_metrics(y_true, y_pred):
    out = {
        "mae": mean_absolute_error(y_true, y_pred),
        "rmse": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "r2": r2_score(y_true, y_pred) if len(y_true) > 1 else float("nan"),
    }
    reduced = (y_true < CFG.REDUCED_EF).astype(int)
    out["auc_ef_lt_50"] = roc_auc_score(reduced, -y_pred) if 0 < reduced.sum() < len(reduced) else float("nan")
    return out


def train_epoch():
    model.train()
    total = 0.0
    for x, y in tqdm(train_loader, desc="train", leave=False):
        x, y = x.to(CFG.DEVICE, non_blocking=True), y.to(CFG.DEVICE, non_blocking=True)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP):
            pred = model(x).squeeze(1)
        loss = criterion(pred.float(), y)
        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        scaler.step(optimizer)
        scaler.update()
        total += loss.item() * len(x)
    return total / len(train_loader.dataset)


@torch.no_grad()
def predict(loader):
    model.eval()
    preds, trues = [], []
    for clips, y in tqdm(loader, desc="eval", leave=False):
        b, k = clips.shape[:2]
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP):
            out = model(clips.flatten(0, 1).to(CFG.DEVICE)).float().view(b, k)
        preds.append(out.mean(1).cpu().numpy())    # average over clips
        trues.append(y.numpy())
    return np.concatenate(trues), np.concatenate(preds)

## Train

In [ ]:
ckpt_path = CFG.OUT_DIR / CFG.CKPT_NAME
best_mae, best_epoch, patience = np.inf, 0, 0
history = []

for epoch in range(1, CFG.EPOCHS + 1):
    t0 = time.time()
    train_mse = train_epoch()
    y_val, p_val = predict(val_loader)
    m = ef_metrics(y_val, p_val)
    scheduler.step()
    history.append({"epoch": epoch, "train_mse": train_mse, **{f"val_{k}": v for k, v in m.items()}})
    print(f"Epoch {epoch:02d} | train MSE {train_mse:.2f} | val MAE {m['mae']:.2f} RMSE {m['rmse']:.2f} "
          f"R2 {m['r2']:.3f} AUC(EF<50) {m['auc_ef_lt_50']:.3f} | {time.time() - t0:.0f}s")

    if m["mae"] < best_mae:
        best_mae, best_epoch, patience = m["mae"], epoch, 0
        torch.save({
            "model_state_dict": model.state_dict(),
            "model_name": "r2plus1d_18",
            "epoch": epoch,
            "best_val_mae": m["mae"],
            "val_metrics": m,
            "frames": CFG.FRAMES,
            "period": CFG.PERIOD,
            "image_size": CFG.IMAGE_SIZE,
            "mean": MEAN.tolist(),
            "std": STD.tolist(),
            "ef_mean": EF_MEAN,
            "pretrained_source": "torchvision Kinetics-400" if CFG.PRETRAINED else "none",
            "dataset": "EchoNet-Dynamic",
        }, ckpt_path)
        print(f"  saved best -> {ckpt_path.name}")
    else:
        patience += 1
        if patience >= CFG.EARLY_STOP_PATIENCE:
            print("Early stopping")
            break

pd.DataFrame(history).to_csv(CFG.OUT_DIR / "echo_training_history.csv", index=False)
print(f"Best epoch {best_epoch} | val MAE {best_mae:.2f}")

## Test set (multi-clip)

In [ ]:
ckpt = torch.load(ckpt_path, map_location=CFG.DEVICE, weights_only=False)
model.load_state_dict(ckpt["model_state_dict"])
y_test, p_test = predict(test_loader)
test_m = ef_metrics(y_test, p_test)
for k, v in test_m.items():
    print(f"Test {k:13s}: {v:.3f}")

pd.DataFrame({"FileName": test_df["FileName"], "EF_true": y_test, "EF_pred": p_test}).to_csv(
    CFG.OUT_DIR / "echo_test_predictions.csv", index=False)

plt.figure(figsize=(5, 5))
plt.scatter(y_test, p_test, s=6, alpha=0.5)
plt.plot([0, 100], [0, 100], "k--", lw=1)
plt.xlabel("True EF (%)"); plt.ylabel("Predicted EF (%)")
plt.title(f"EchoNet-Dynamic test | MAE {test_m['mae']:.2f}")
plt.xlim(0, 100); plt.ylim(0, 100)
plt.show()

## Inference helper

What the CardioGraph backend needs: one AVI in, EF % and category out.
Categories follow the heart-failure EF ranges: reduced <= 40 %, mildly reduced 41-49 %, normal >= 50 %.

In [ ]:
def ef_category(ef):
    if ef <= 40:
        return "Reduced (HFrEF range, <=40%)"
    if ef < 50:
        return "Mildly reduced (41-49%)"
    return "Normal (>=50%)"


@torch.no_grad()
def predict_ef(video_path, clips=CFG.TEST_CLIPS):
    model.eval()
    video = load_video(video_path)
    x = torch.stack([clip_from(video, s) for s in clip_starts(len(video), clips)]).to(CFG.DEVICE)
    ef = float(model(x).float().mean())
    return {"ef": round(ef, 1), "category": ef_category(ef), "frames": len(video)}


row = test_df.iloc[0]
print(row["FileName"], "| true EF", round(row["EF"], 1), "->", predict_ef(CFG.VIDEO_DIR / row["FileName"]))